# Governed tool-calling agent with a kill switch

This notebook shows a LangChain-style tool-calling agent wrapped with **TealTiger** deterministic governance, plus an **emergency kill switch (FREEZE)** that instantly stops all agent actions without restarting the service.

Production agents sometimes go haywire: calling tools in loops, spending budget, or executing dangerous actions. Operators need to freeze an agent *right now* and unfreeze it once the situation is understood.

You will build an agent that:

- exposes 5 tools (`search`, `calculator`, `email`, `file_write`, `database`),
- allows only `search` and `calculator` via a tool allowlist,
- unconditionally blocks `file_write` and `database` via FREEZE rules,
- rate-limits to 10 tool calls per minute,
- can be frozen mid-session (kill switch) and unfrozen to resume.

No API keys are required. The tools are mocked and the governance engine is fully deterministic (no LLM in the governance path).

## Install

In [ ]:
%pip install -U -q "langchain-tealtiger>=0.4.5"

## Define mock tools

Each tool is a plain Python function. No network calls, so you can run this notebook anywhere.

In [ ]:
def search(query: str) -> str:
    return f"[search results for {query!r}]"

def calculator(expression: str) -> str:
    return f"{expression} = {eval(expression, {'__builtins__': {}}, {})}"

def email(to: str, body: str) -> str:
    return f"[email sent to {to}]"

def file_write(path: str, content: str) -> str:
    return f"[wrote {len(content)} bytes to {path}]"

def database(query: str) -> str:
    return f"[db executed: {query}]"

TOOLS = {
    "search": search,
    "calculator": calculator,
    "email": email,
    "file_write": file_write,
    "database": database,
}
print("Tools:", list(TOOLS))

## Configure governance

The `TealTigerMiddleware` is what you attach to a real LangChain agent via `create_agent(..., middleware=[...])`. In this notebook you drive its governance engine directly through a tiny agent loop so the demo runs without a model.

Policies:

- **tool_allowlist** — only `search` and `calculator` may run.
- **freeze_tools** — `file_write` and `database` are immutable denies, blocked regardless of mode.
- **rate_limit** — at most 10 tool calls per minute.

In [ ]:
from langchain_tealtiger import TealTigerMiddleware

middleware = TealTigerMiddleware(
    policies=[
        {"type": "tool_allowlist", "tools": ["search", "calculator"]},
        {"type": "rate_limit", "max_calls": 10, "window": "1m"},
    ],
    freeze_tools=["file_write", "database"],
    mode="ENFORCE",
)

# The governance engine used by the middleware. In a real agent you never call
# this directly; the middleware invokes it inside wrap_tool_call().
engine = middleware._engine
print("Middleware ready. Frozen tools:", middleware.frozen_tools())

## A minimal governed agent loop

This helper simulates what happens inside `wrap_tool_call`: every tool call is evaluated by the governance engine first. A `DENY` short-circuits execution and returns a structured decision instead of running the tool.

In [ ]:
from langchain_tealtiger._types import GovernanceAction

def call_tool(name: str, **args):
    """Governed tool call: evaluate, then run only if allowed."""
    decision = engine.evaluate(tool_name=name, tool_args=args)
    if decision.action == GovernanceAction.DENY:
        print(f"  DENY  {name:<12} reason={decision.reason}")
        print(f"        reason_codes={decision.reason_codes}")
        return None
    result = TOOLS[name](**args)
    print(f"  ALLOW {name:<12} -> {result}")
    return result

## 1. Normal operation

Allowlisted tools run. Non-allowlisted tools (`email`) are denied. Frozen tools (`file_write`, `database`) are denied even though they exist.

In [ ]:
print("Normal operation:")
call_tool("search", query="tealtiger governance")
call_tool("calculator", expression="2 + 2")
call_tool("email", to="ops@example.com", body="hi")        # not in allowlist
call_tool("file_write", path="/etc/passwd", content="x")   # frozen
call_tool("database", query="DROP TABLE users")            # frozen

## 2. Rate limiting

The agent is capped at 10 tool calls per minute. The 11th allowed call is throttled with a structured `RATE_LIMIT_EXCEEDED` decision.

In [ ]:
engine.reset_session()  # clear the call counter for a clean demo
print("Firing 12 search calls (limit is 10/min):")
for i in range(12):
    print(f"call {i + 1:>2}:", end=" ")
    call_tool("search", query=f"q{i}")

## 3. Kill switch — freeze all tools mid-session

Something looks wrong. An operator engages the emergency kill switch. Every tool call is now denied with a `KILL_SWITCH` reason code, regardless of allowlist or mode. No service restart needed.

In [ ]:
engine.reset_session()

print("Before kill switch:")
call_tool("search", query="before")

print("\n>>> middleware.freeze_all()  # KILL SWITCH ENGAGED\n")
middleware.freeze_all()
print("is_frozen():", middleware.is_frozen())

print("\nDuring freeze (all tools blocked):")
call_tool("search", query="during")
call_tool("calculator", expression="1 + 1")

## 4. Unfreeze and resume

Once the situation is understood, the operator releases the kill switch. Normal governance resumes: allowlisted tools work again, and the original FREEZE rules (`file_write`, `database`) are still enforced.

In [ ]:
print(">>> middleware.unfreeze_all()  # KILL SWITCH RELEASED\n")
middleware.unfreeze_all()
print("is_frozen():", middleware.is_frozen())

print("\nAfter unfreeze:")
call_tool("search", query="after")
call_tool("calculator", expression="3 * 7")
call_tool("database", query="SELECT 1")   # still frozen by original policy

## 5. Targeted freeze — one tool at a time

Beyond the all-or-nothing kill switch, you can freeze and unfreeze individual tools at runtime with `freeze(...)` / `unfreeze(...)`.

In [ ]:
engine.reset_session()

print(">>> middleware.freeze('calculator')\n")
middleware.freeze("calculator")
print("frozen_tools():", middleware.frozen_tools())
call_tool("search", query="still ok")
call_tool("calculator", expression="9 - 4")   # now frozen

print("\n>>> middleware.unfreeze('calculator')\n")
middleware.unfreeze("calculator")
call_tool("calculator", expression="9 - 4")   # restored

## Wiring into a real LangChain agent

In production you attach the middleware to `create_agent` and expose the kill switch to your operations tooling (an admin endpoint, a CLI, a dashboard button). The governance runs inside `wrap_tool_call`, so no agent code changes are needed.

```python
from langchain.agents import create_agent
from langchain_tealtiger import TealTigerMiddleware

middleware = TealTigerMiddleware(
    policies=[
        {"type": "tool_allowlist", "tools": ["search", "calculator"]},
        {"type": "rate_limit", "max_calls": 10, "window": "1m"},
    ],
    freeze_tools=["file_write", "database"],
    mode="ENFORCE",
)

agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search, calculator, email, file_write, database],
    middleware=[middleware],
)

# Emergency controls, callable from anywhere that holds the middleware:
middleware.freeze_all()      # halt everything
middleware.unfreeze_all()    # resume
middleware.freeze("database")   # halt one tool
middleware.is_frozen("database")
```

## What's next

- **Governed SQL agent** — table-level access control with `table_allowlist` / `table_blocklist`.
- **PII redaction** — scan tool outputs and model responses with `pii_block` / `secret_detection` policies.
- **Audit evidence** — export governance decisions via `middleware._engine.evidence()` for SARIF / JUnit reporting.

See the [langchain-tealtiger docs](https://pypi.org/project/langchain-tealtiger/) for the full policy reference.